# Exploratory Data Analysis & Historical Base-Rate Analysis - Project 2
> **Governance Note:** Examines feature distributions and historical default base rates across age cohorts (`Age < 30` vs. `Age >= 30`). Note that the EEOC Four-Fifths Rule ($0.80$–$1.25$) applies to a model's selection/approval rates, whereas differences in raw dataset default rates are descriptive base-rate differences.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(PROJECT_ROOT / "data/processed/train.parquet")
print(f"Training Partition Shape: {df.shape}")

In [ ]:
numerical_cols = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio",
]
categorical_cols = [
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner",
]
print(df[numerical_cols].describe().round(2))

In [ ]:
# Descriptive Historical Default Base-Rate Comparison (Not a 4/5ths selection-rate test)
df_eda = df.copy()
df_eda["age_cohort"] = df_eda["Age"].apply(
    lambda x: "UNDER_30" if x < 30 else "30_AND_OVER"
)
cohort_default_rates = df_eda.groupby("age_cohort")["Default"].mean()
young_def = float(cohort_default_rates["UNDER_30"])
mature_def = float(cohort_default_rates["30_AND_OVER"])
base_rate_ratio = young_def / mature_def if mature_def > 0 else 0.0
print(
    f"Descriptive Default Base Rates -> UNDER_30: {young_def:.4f} | 30_AND_OVER: {mature_def:.4f}"
)
print(f"Historical Default Base-Rate Ratio (Descriptive): {base_rate_ratio:.4f}")
print(
    "Note: An unmitigated model trained on raw base-rate differences may under-approve younger applicants; sample reweighting is applied in src/training/train.py."
)